In [12]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [13]:
import time
import random
import numpy as np
import pandas as pd

from algorithms.graph import (
    is_independent,
    solution_size,
    generate_random_graph
)

from algorithms.brute_force import brute_force_mis
from algorithms.greedy import greedy_mis
from algorithms.genetic_algorithm import (
    Individual,
    selection,
    crossover,
    mutation,
    genetic_algorithm
)
from algorithms.tabu_search import (
    get_neighbors,
    get_conflicting_vertices,
    is_tabu,
    is_move_tabu,
    update_tabu_list,
    generate_initial_solution,
    classify_neighbors,
    select_move,
    apply_move,
    update_tabu_after_move,
    tabu_search
)

In [14]:
graph = generate_random_graph(8, 0.3)

print(graph)


[[0 0 0 0 1 0 0 1]
 [0 0 0 1 0 0 1 1]
 [0 0 0 1 0 1 0 0]
 [0 1 1 0 1 0 0 0]
 [1 0 0 1 0 0 1 1]
 [0 0 1 0 0 0 1 0]
 [0 1 0 0 1 1 0 0]
 [1 1 0 0 1 0 0 0]]


#### Testiranje Brute Force algoritma

In [15]:
start = time.perf_counter()

brute_solution = brute_force_mis(graph)

end = time.perf_counter()

print("Brute Force:")
print("Rešenje:", brute_solution)
print("Fitness:", len(brute_solution))
print("Vreme:", end - start)
print("Validno:", is_independent(graph, brute_solution))

Brute Force:
Rešenje: [0, 1, 2]
Fitness: 3
Vreme: 0.0004642529966076836
Validno: True


#### Testiranje Greedy algoritma

In [16]:
start = time.perf_counter()

greedy_solution = greedy_mis(graph)

end = time.perf_counter()

print("Greedy:")
print("Rešenje:", greedy_solution)
print("Fitness:", len(greedy_solution))
print("Vreme:", end - start)
print("Validno:", is_independent(graph, greedy_solution))

Greedy:
Rešenje: [0, 1, 2]
Fitness: 3
Vreme: 7.861199992476031e-05
Validno: True


#### Testiranje Genetic algoritma

In [17]:
start = time.perf_counter()

ga_solution = genetic_algorithm(
    graph,
    population_size=50,
    max_evaluations=20000,
    crossover_rate=0.6
)

ga_time = time.perf_counter() - start

print("Genetic Algorithm:")
print("Rešenje:", ga_solution)
print("Fitness:", len(ga_solution))
print("Vreme:", ga_time)
print("Validno:", is_independent(graph, ga_solution))

Genetic Algorithm:
Rešenje: [0, 1, 2]
Fitness: 3
Vreme: 0.44744725900091
Validno: True


#### Testiranje Tabu Search algoritma

In [18]:
start = time.perf_counter()

tabu_solution = tabu_search(
    graph,
    iterations=100,
    tabu_tenure=5
)

end = time.perf_counter()

print("Tabu Search:")
print("Rešenje:", tabu_solution)
print("Fitness:", len(tabu_solution))
print("Vreme:", end - start)
print("Validno:", is_independent(graph, tabu_solution))

Tabu Search:
Rešenje: [2, 0, 6]
Fitness: 3
Vreme: 0.0010321709996787831
Validno: True


#### Poređenje rezultata

In [20]:
def run_all_algorithms(graph):
    
    results = []

    # Brute Force
    start = time.perf_counter()
    solution = brute_force_mis(graph)
    elapsed = time.perf_counter() - start

    results.append({
        "Algorithm": "Brute Force",
        "Solution": solution,
        "Size": len(solution),
        "Time (s)": elapsed,
        "Valid": is_independent(graph, solution)
    })

    # Greedy
    start = time.perf_counter()
    solution = greedy_mis(graph)
    elapsed = time.perf_counter() - start

    results.append({
        "Algorithm": "Greedy",
        "Solution": solution,
        "Size": len(solution),
        "Time (s)": elapsed,
        "Valid": is_independent(graph, solution)
    })

    # Genetic Algorithm
    start = time.perf_counter()
    solution = genetic_algorithm(
        graph,
        population_size=50,
        max_evaluations=20000,
        crossover_rate=0.6
    )
    elapsed = time.perf_counter() - start

    results.append({
        "Algorithm": "Genetic Algorithm",
        "Solution": solution,
        "Size": len(solution),
        "Time (s)": elapsed,
        "Valid": is_independent(graph, solution)
    })

    # Tabu Search
    start = time.perf_counter()
    solution = tabu_search(
        graph,
        iterations=100,
        tabu_tenure=5
    )
    elapsed = time.perf_counter() - start

    results.append({
        "Algorithm": "Tabu Search",
        "Solution": solution,
        "Size": len(solution),
        "Time (s)": elapsed,
        "Valid": is_independent(graph, solution)
    })

    return pd.DataFrame(results)

results = run_all_algorithms(graph)

optimal_size = results.loc[
    results["Algorithm"] == "Brute Force",
    "Size"
].iloc[0]

results["Gap (%)"] = (
    (optimal_size - results["Size"])
    / optimal_size
    * 100
)

results

,Algorithm,Solution,Size,Time (s),Valid,Gap (%)
0,Brute Force,"[0, 1, 2]",3,0.000285,True,0.0
1,Greedy,"[0, 1, 2]",3,0.000038,True,0.0
2,Genetic Algorithm,"[2, 6, 7]",3,0.437963,True,0.0
3,Tabu Search,"[5, 7, 3]",3,0.000830,True,0.0
